In [ ]:
import os
import pandas as pd
import numpy as np
import json
import shutil
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

## Hybrid Model Testing

Testing the hybrid model using soft voting classifier combining DenseNet201 (US images) and XGBoost (clinical data)

In [ ]:
print("\n" + "="*80)
print("LOADING TRAINED MODELS")
print("="*80)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load DenseNet201 model for US images
densenet_model_path = './Models/Ultrasound/Densenet201/densenet201_full_model.pt'
print(f"\nLoading DenseNet201 from: {densenet_model_path}")
densenet_model = torch.load(densenet_model_path, map_location=device, weights_only=False)
densenet_model.eval()
print("✅ DenseNet201 loaded successfully")

# Load XGBoost model for clinical data
xgboost_model_path = './Models/Clinical_Data/XGBoost/xgboost_clinical_model.json'
print(f"\nLoading XGBoost from: {xgboost_model_path}")
xgboost_model = xgb.Booster()
xgboost_model.load_model(xgboost_model_path)
print("✅ XGBoost loaded successfully")

In [ ]:
# Load model accuracies for weighting
print("\n" + "="*80)
print("LOADING MODEL ACCURACIES")
print("="*80)

# Load XGBoost metrics
with open('./Models/Clinical_Data/XGBoost/model_metrics.json', 'r') as f:
    xgboost_metrics = json.load(f)
    
# DenseNet201 achieved 97% validation accuracy during training
xgboost_accuracy = xgboost_metrics.get('test_accuracy', 0.45)
densenet_accuracy = 0.97  # From DenseNet201 training results

print(f"XGBoost (Clinical) Accuracy: {xgboost_accuracy:.4f}")
print(f"DenseNet201 (US Image) Accuracy: {densenet_accuracy:.4f}")

# Calculate weights based on accuracies
A_b = xgboost_accuracy  # Blood test (clinical) accuracy
A_i = densenet_accuracy  # Image accuracy

w_b = A_b / (A_b + A_i)  # Weight for blood test model
w_i = A_i / (A_b + A_i)  # Weight for image model

print(f"\nCalculated Weights:")
print(f"  Clinical (XGBoost) weight: {w_b:.4f}")
print(f"  Image (DenseNet) weight: {w_i:.4f}")

In [ ]:
def predict_hybrid(us_image_path, clinical_features, densenet_model, xgboost_model, transform, device, w_b, w_i):
    """
    Perform hybrid prediction using soft voting classifier
    
    Parameters:
    - us_image_path: Path to ultrasound image
    - clinical_features: Clinical data features (pandas Series or dict)
    - densenet_model: Trained DenseNet model
    - xgboost_model: Trained XGBoost model
    - transform: Image preprocessing transform
    - device: torch device
    - w_b: Weight for blood test (clinical) model
    - w_i: Weight for image model
    
    Returns:
    - final_prediction: Predicted class
    - P_hybrid: Final hybrid probabilities for all classes
    - P_b: Clinical model probabilities
    - P_i: Image model probabilities
    """
    
    # 1. Get DenseNet (image) prediction probabilities
    image = Image.open(us_image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        image_output = densenet_model(image_tensor)
        P_i = torch.softmax(image_output, dim=1).cpu().numpy()[0]  # Probabilities for each class
    
    # 2. Get XGBoost (clinical) prediction probabilities
    # Prepare clinical features as DMatrix
    clinical_features_df = pd.DataFrame([clinical_features])
    # Remove non-feature columns if present
    feature_columns = [col for col in clinical_features_df.columns 
                      if col not in ['Stage', 'us_image', 'folder']]
    clinical_features_clean = clinical_features_df[feature_columns]
    
    dmatrix = xgb.DMatrix(clinical_features_clean)
    P_b = xgboost_model.predict(dmatrix)[0]  # Probabilities for each class
    
    # 3. Calculate weighted hybrid probabilities using soft voting
    # P_hybrid,c = (w_b * P_b,c) + (w_i * P_i,c) for each class c
    P_hybrid = (w_b * P_b) + (w_i * P_i)
    
    # 4. Final prediction is the class with highest hybrid probability
    final_prediction = np.argmax(P_hybrid)
    
    return final_prediction, P_hybrid, P_b, P_i

print("✅ Hybrid prediction function defined")

In [ ]:
# Run predictions on test set
print("\n" + "="*80)
print("RUNNING HYBRID MODEL PREDICTIONS ON TEST SET")
print("="*80)

# Load test clinical data
test_clinical_path = './Hybrid_Test_Dataset/test_clinical_data.csv'
test_clinical_df = pd.read_csv(test_clinical_path)

print(f"Loaded {len(test_clinical_df)} test samples")

# Store predictions
predictions = []
true_labels = []
all_hybrid_probs = []
all_clinical_probs = []
all_image_probs = []

print("\nProcessing test samples...")
for idx, row in test_clinical_df.iterrows():
    # Get image path
    stage_num = int(row['Stage'])
    us_image_name = row['us_image']
    us_image_path = os.path.join('./Hybrid_Test_Dataset', f'Stage_{stage_num}', 'US_Images', us_image_name)
    
    # Get clinical features
    clinical_features = row.drop(['us_image', 'folder'])
    
    # Get true label (0-indexed)
    true_label = int(row['Stage']) - 1
    
    # Make prediction
    pred, P_hybrid, P_b, P_i = predict_hybrid(
        us_image_path, clinical_features, 
        densenet_model, xgboost_model, 
        transform, device, w_b, w_i
    )
    
    predictions.append(pred)
    true_labels.append(true_label)
    all_hybrid_probs.append(P_hybrid)
    all_clinical_probs.append(P_b)
    all_image_probs.append(P_i)
    
    if (idx + 1) % 10 == 0:
        print(f"  Processed {idx + 1}/{len(test_clinical_df)} samples")

print(f"\n✅ Completed predictions for all {len(test_clinical_df)} test samples")

In [ ]:
# Calculate and display results
print("\n" + "="*80)
print("HYBRID MODEL EVALUATION RESULTS")
print("="*80)

# Calculate accuracy
hybrid_accuracy = accuracy_score(true_labels, predictions)
print(f"\nHybrid Model Accuracy: {hybrid_accuracy:.4f} ({hybrid_accuracy*100:.2f}%)")

# Compare with individual model accuracies
print(f"\nModel Comparison:")
print(f"  XGBoost (Clinical) Accuracy: {xgboost_accuracy:.4f} ({xgboost_accuracy*100:.2f}%)")
print(f"  DenseNet201 (Image) Accuracy: {densenet_accuracy:.4f} ({densenet_accuracy*100:.2f}%)")
print(f"  Hybrid Model Accuracy: {hybrid_accuracy:.4f} ({hybrid_accuracy*100:.2f}%)")
print(f"\n  Improvement over Clinical-only: {(hybrid_accuracy - xgboost_accuracy)*100:.2f}%")
print(f"  Improvement over Image-only: {(hybrid_accuracy - densenet_accuracy)*100:.2f}%")

# Classification report
print("\n" + "="*80)
print("DETAILED CLASSIFICATION REPORT")
print("="*80)
class_names = ['Stage 1', 'Stage 2', 'Stage 3', 'Stage 4']
print("\n" + classification_report(true_labels, predictions, target_names=class_names, digits=4))

In [ ]:
# Visualize confusion matrix
print("\n" + "="*80)
print("CONFUSION MATRIX")
print("="*80)

cm = confusion_matrix(true_labels, predictions)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Hybrid Model - Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

print("\nConfusion Matrix:")
print(cm)

In [ ]:
# Visualize example predictions with probability distributions
print("\n" + "="*80)
print("EXAMPLE PREDICTIONS WITH PROBABILITY DISTRIBUTIONS")
print("="*80)

# Show first 5 test samples
num_examples = min(5, len(test_clinical_df))
fig, axes = plt.subplots(num_examples, 4, figsize=(16, 4*num_examples))

if num_examples == 1:
    axes = axes.reshape(1, -1)

for i in range(num_examples):
    row = test_clinical_df.iloc[i]
    stage_num = int(row['Stage'])
    us_image_name = row['us_image']
    us_image_path = os.path.join('./Hybrid_Test_Dataset', f'Stage_{stage_num}', 'US_Images', us_image_name)
    
    # Load and display image
    img = Image.open(us_image_path)
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f'Sample {i+1}\nTrue: Stage {stage_num}', fontweight='bold')
    axes[i, 0].axis('off')
    
    # Plot clinical model probabilities
    axes[i, 1].bar(class_names, all_clinical_probs[i], color='skyblue', alpha=0.7)
    axes[i, 1].set_title(f'Clinical Model (XGBoost)\nPred: Stage {predictions[i]+1}')
    axes[i, 1].set_ylabel('Probability')
    axes[i, 1].set_ylim([0, 1])
    axes[i, 1].tick_params(axis='x', rotation=45)
    
    # Plot image model probabilities
    axes[i, 2].bar(class_names, all_image_probs[i], color='lightcoral', alpha=0.7)
    axes[i, 2].set_title(f'Image Model (DenseNet)\nPred: Stage {np.argmax(all_image_probs[i])+1}')
    axes[i, 2].set_ylabel('Probability')
    axes[i, 2].set_ylim([0, 1])
    axes[i, 2].tick_params(axis='x', rotation=45)
    
    # Plot hybrid model probabilities
    axes[i, 3].bar(class_names, all_hybrid_probs[i], color='lightgreen', alpha=0.7)
    axes[i, 3].set_title(f'Hybrid Model\nPred: Stage {predictions[i]+1}', fontweight='bold')
    axes[i, 3].set_ylabel('Probability')
    axes[i, 3].set_ylim([0, 1])
    axes[i, 3].tick_params(axis='x', rotation=45)
    
    # Highlight correct/incorrect prediction
    if predictions[i] == (stage_num - 1):
        axes[i, 3].patch.set_edgecolor('green')
        axes[i, 3].patch.set_linewidth(3)
    else:
        axes[i, 3].patch.set_edgecolor('red')
        axes[i, 3].patch.set_linewidth(3)

plt.tight_layout()
plt.show()

In [ ]:
# # Save hybrid model results
# print("\n" + "="*80)
# print("SAVING HYBRID MODEL RESULTS")
# print("="*80)

# # Create results directory
# results_dir = './Models/Hybrid'
# os.makedirs(results_dir, exist_ok=True)

# # Save detailed results
# results = {
#     'model_type': 'Soft Voting Classifier (DenseNet201 + XGBoost)',
#     'test_accuracy': float(hybrid_accuracy),
#     'xgboost_accuracy': float(xgboost_accuracy),
#     'densenet_accuracy': float(densenet_accuracy),
#     'weights': {
#         'clinical_weight': float(w_b),
#         'image_weight': float(w_i)
#     },
#     'improvement': {
#         'over_clinical': float(hybrid_accuracy - xgboost_accuracy),
#         'over_image': float(hybrid_accuracy - densenet_accuracy)
#     },
#     'confusion_matrix': cm.tolist(),
#     'predictions': [int(p) for p in predictions],
#     'true_labels': [int(t) for t in true_labels]
# }

# results_path = os.path.join(results_dir, 'hybrid_model_results.json')
# with open(results_path, 'w') as f:
#     json.dump(results, f, indent=4)

# print(f"✅ Results saved to: {results_path}")

# # Save predictions CSV
# predictions_df = test_clinical_df.copy()
# predictions_df['true_label'] = [t + 1 for t in true_labels]
# predictions_df['predicted_label'] = [p + 1 for p in predictions]
# predictions_df['correct'] = predictions_df['true_label'] == predictions_df['predicted_label']

# # Add probability columns
# for i, class_name in enumerate(class_names):
#     predictions_df[f'hybrid_prob_{class_name}'] = [probs[i] for probs in all_hybrid_probs]
#     predictions_df[f'clinical_prob_{class_name}'] = [probs[i] for probs in all_clinical_probs]
#     predictions_df[f'image_prob_{class_name}'] = [probs[i] for probs in all_image_probs]

# predictions_csv = os.path.join(results_dir, 'hybrid_predictions.csv')
# predictions_df.to_csv(predictions_csv, index=False)
# print(f"✅ Predictions saved to: {predictions_csv}")

# print("\n" + "="*80)
# print("HYBRID MODEL TESTING COMPLETE")
# print("="*80)